In [ ]:
from random import choice
from copy import copy
from typing import Tuple, Callable

In [ ]:
def successors_by_predecessors(predecessors: list[list[int]]):
    size = len(predecessors)
    return [[succ for succ in range(size) if i in predecessors[succ]] for i in range(size)]

def calculate_critical_times(
        duration: list[int],
        predecessors: list[list[int]],
        successors: list[list[int]] = None) -> Tuple[list[int], list[int]]:
    """Расчёт ранних времён начала и поздних времён завершения работ

    Args:
        duration (list[int]): _description_
        predecessors (list[list[int]]): списки предшествующих работ
        successors (list[list[int]]): списки последующих работ

    Raises:
        ValueError:

    Returns:
        Tuple[list[int], list[int]]: ранние времена начала и поздние времена завершения
    """
    if not successors:
        successors = successors_by_predecessors(predecessors)
    if len(duration) != len(predecessors) or len(predecessors) != len(successors):
        raise ValueError("Invalid data")
    earliest_start = [0 for _ in duration]
    latest_finish = [0 for _ in duration]

    def _calc_earliest_start(index):
        if index:
            earliest_start[index] = max(_calc_earliest_start(pred) + duration[pred] for pred in predecessors[index])
        return earliest_start[index]

    def _calc_latest_finish(index):
        if index == len(successors) - 1:
            latest_finish[index] = earliest_start[index]
        else:
            latest_finish[index] = min(_calc_latest_finish(succ) - duration[succ] for succ in successors[index])
        return latest_finish[index]

    _calc_earliest_start(len(predecessors) - 1)
    _calc_latest_finish(0)

    return earliest_start, latest_finish

class TimeCapacityNode:
    """
    Вспомогательная структура связного списка моментов времени и
    имеющихся в них остаточных запасов ресурсов
    """
    def __init__(self, time: int, capacity: list[int]):
        self.time = time
        self.capacity = capacity
        self.next = None
        self.prev = None

    def insert_after(self, time: int) -> 'TimeCapacityNode':
        if time <= self.time:
            raise ValueError("Invalid time")

        new_node = self.__class__(time, copy(self.capacity))
        new_node.prev = self
        new_node.next = self.next
        if self.next:
            self.next.prev = new_node
        self.next = new_node
        return new_node

    def find_first(self, time: int) -> 'TimeCapacityNode':
        node = self
        while node.time < time:
            if node.next:
                node = node.next
            else:
                return node
        return node.prev

    def enough_resources(self, demand: list[int]) -> bool:
        return all(self.capacity[i] >= demand[i] for i in range(len(self.capacity)))

    def consume(self, demand: list[int]) -> None:
        for i in range(len(self.capacity)):
            self.capacity[i] -= demand[i]

class ActivityListDecoder:
    def decode(
            self,
            activity_list: list[int],
            duration: list[int],
            predecessors: list[list[int]],
            renewable_demands: list[list[int]],
            renewable_capacity: list[int]) -> list[int]:
        """
        Последовательная схема генерации расписания для декодирования Activity List.

        Args:
            activity_list (list[int]): закодированное решение (Activity List)
            duration (list[int]): продолжительности работ
            predecessors (list[list[int]]): списки предшествующих работ
            renewable_demands (list[list[int]]): затраты неисчерпаемых ресурсов
            renewable_capacity (list[int]): запасы неисчерпаемых ресурсов

        Raises:
            ValueError: выбрасывается при нарушении связей предшествования

        Returns:
            list[int]: времена начала работ
        """
        count = len(activity_list)
        root_node = TimeCapacityNode(0, copy(renewable_capacity)) # Связный список моментов изменения запаса ресурсов
        starts = [0] * count
        finish_nodes = [None] * count
        finish_nodes[0] = root_node

        for i in activity_list:
            # Работа может начаться не раньше, чем её последняя предшественница
            start_node = root_node
            for pred in predecessors[i]:
                if not finish_nodes[pred]:
                    raise ValueError("Invalid activity list")
                if finish_nodes[pred].time > start_node.time:
                    start_node = finish_nodes[pred]
            # Ищем такую позицию для начала, при которой не нарушатся ресурсные ограничения
            start_node, last_node, finish_node, finish_time = self._find_position(
                start_node, duration[i], renewable_demands[i]
            )
            starts[i] = start_node.time
            if not finish_node or finish_node.time != finish_time:
                finish_node = last_node.insert_after(finish_time)
            finish_nodes[i] = finish_node
            # Обновляем доступное число ресурсов в моменты времени, затрагиваемые данной работой
            self._consume(start_node, finish_node, renewable_demands[i])

        return starts

    def _consume(
            self, start_node: TimeCapacityNode,
            finish_node: TimeCapacityNode,
            demand: list[int]) -> None:
        node = start_node
        while node != finish_node:
            node.consume(demand)
            node = node.next

    def _find_position(
            self, start_node: int, duration: int, demand: list[int]
            ) -> Tuple[TimeCapacityNode, TimeCapacityNode, TimeCapacityNode, int]:
        if not duration:
            return (start_node, start_node, start_node, start_node.time)

        finish_time = start_node.time + duration
        t = start_node.find_first(finish_time)
        last_node = t
        t_test = start_node

        while t != t_test.prev:
            if t.enough_resources(demand):
                t = t.prev
            else:
                start_node = t.next
                finish_time = start_node.time + duration
                if last_node.next:
                    t_test = last_node.next
                    last_node = t_test.find_first(finish_time)
                    t = last_node
                else:
                    break

        return (start_node, last_node, last_node.next, finish_time)

class ActivityListSampler:
    def __init__(
            self,
            predecessors: list[list[int]],
            successors: list[list[int]] = None) -> None:
        """
        Args:
            predecessors (list[list[int]]): списки предшествующих работ
            successors (list[list[int]]): списки последующих работ
        """
        self.predecessors = predecessors
        self.size = len(predecessors)
        if not successors:
            successors = successors_by_predecessors(predecessors)
        self.successors = successors

    def _generate(self, func: Callable = None) -> list[int]:
        result = []
        ramain_predecessors = [set(pred) for pred in self.predecessors]
        ready_set = [i for i in range(self.size) if not self.predecessors[i]]

        for _ in range(self.size):
            if not ready_set:
                raise ValueError("Incorrect project network")

            next_activity = func(ready_set) if func else choice(ready_set)
            ready_set.remove(next_activity)
            result.append(next_activity)

            for successor in self.successors[next_activity]:
                ramain_predecessors[successor].remove(next_activity)
                if not ramain_predecessors[successor]:
                    ready_set.append(successor)

        return result

    def generate_random(self) -> list[int]:
        """
        Генерация Activity List случайным образом

        Returns:
            list[int]: Activity List
        """
        return self._generate()

    def generate_by_max_rule(self, rule: Callable) -> list[int]:
        """
        Генерация Activity List упорядочивая по убыванию метрики

        Returns:
            list[int]: Activity List
        """
        def func(data: list[int]):
            return max(data, key=rule)
        return self._generate(func)

    def generate_by_min_rule(self, rule: Callable) -> list[int]:
        """
        Генерация Activity List упорядочивая по возрастанию метрики

        Returns:
            list[int]: Activity List
        """
        def func(data: list[int]):
            return min(data, key=rule)
        return self._generate(func)

Пример на диске

In [ ]:
predecessors = [[], [0], [1], [1], [1], [4], [2,3,5], [6]]
# Длительности
durations = [0, 5, 15, 5, 8, 2, 8, 0]

In [6]:
# Рассчёт ранних времён начала и поздних времён окончания
earliest_start, latest_finish = calculate_critical_times(durations, predecessors)

In [7]:
earliest_start

[0, 0, 5, 5, 5, 13, 20, 28]

In [8]:
latest_finish

[0, 5, 20, 20, 18, 20, 28, 28]

In [9]:
# Ресурсные затраты
renewable_demands = [[0, 0], [2, 2], [2, 3], [3, 1], [4, 0], [2, 2], [0, 4], [0, 0]]
renewable_capacities = [5, 6]

In [10]:
sampler = ActivityListSampler(predecessors)
decoder = ActivityListDecoder()

In [11]:
# Генерируем случайный порядок
random_activity_list = sampler.generate_random()
start_times = decoder.decode(random_activity_list, durations, predecessors, renewable_demands, renewable_capacities)

In [12]:
random_activity_list

[0, 1, 3, 4, 5, 2, 6, 7]

In [13]:
start_times

[0, 0, 18, 5, 10, 18, 33, 41]

In [14]:
# Генерируем порядок в соответствии с правилом (по возрастанию поздних времен конца)
heuristic_activity_list = sampler.generate_by_min_rule(lambda j: latest_finish[j])
start_times2 = decoder.decode(random_activity_list, durations, predecessors, renewable_demands, renewable_capacities)

In [15]:
heuristic_activity_list

[0, 1, 4, 2, 3, 5, 6, 7]

In [16]:
start_times2

[0, 0, 18, 5, 10, 18, 33, 41]

Наш проект - "Мобильное приложение для ведения бюджета" - со "случайным" решением + эвристика из шаблона

In [42]:
predecessors = [
    [],
    [0],        # 0 (фиктивная начальная работа)
    [1],        # 1 - Реализовать экран регистрации
    [2],        # 2 - Реализовать сервис отправки и проверки кода подтверждения по email
    [3],        # 3 - Разработать экран ввода и подтверждения кода
    [2],        # 4 - Реализовать экран профиля пользователя
    [1],        # 5 - Разработать механизм аутентификации
    [1],        # 6 - Реализовать экран "Мои финансы"
    [7],        # 7 - Разработать экран добавления разового дохода
    [7],        # 8 - Разработать экран добавления разового расхода
    [7],        # 9 - Функционал выбора регулярного дохода/расхода
    [1],        # 10 - Разработать экран "Активы и пассивы"
    [10],       # 11 - Реализовать систему тегов
    [12],       # 12 - Добавить тег в процессе фиксации
    [10],       # 13 - Функционал создания напоминания
    [4, 5, 6],  # 14 - Реализовать систему пуш-уведомлений
    [11],       # 15 - Функционал планирования крупной цели
    [15],       # 16 - Ежемесячное уведомление о бронировании
    [8, 9],     # 17 - Предупреждение о превышении расходов
    [12],       # 18 - Раздел "Аналитика"
    [19],       # 19 - Визуализация: круговая диаграмма
    [17, 18, 19, 20],      # 20 - Визуализация: график доходов/расходов
    [14, 16],              # 21 - Отображение прогресса по целям
    [22],                  # 22 - Функционал "Анализа спонтанных трат"
    [13, 21, 23],          # 23 - Раздел "Обучение"
    [24]                  # 24 - Экран чтения статей
]

In [43]:
durations = [
    0,   # 0 (фиктивная начальная)
    5,   # 1 - Экран регистрации
    3,   # 2 - Сервис email
    4,   # 3 - Экран подтверждения кода
    3,   # 4 - Профиль пользователя
    4,   # 5 - Аутентификация
    2,   # 6 - Экран "Мои финансы"
    5,   # 7 - Добавление дохода
    5,   # 8 - Добавление расхода
    3,   # 9 - Выбор регулярных операций
    6,   # 10 - Активы и пассивы
    4,   # 11 - Система тегов
    2,   # 12 - Добавить тег в процессе
    3,   # 13 - Создание напоминания
    7,   # 14 - Пуш-уведомления
    8,   # 15 - Планирование крупной цели
    2,   # 16 - Ежемесячное уведомление
    3,   # 17 - Предупреждение о превышении
    10,  # 18 - Раздел "Аналитика"
    6,   # 19 - Круговая диаграмма
    7,   # 20 - График доходов/расходов
    4,   # 21 - Прогресс по целям
    5,   # 22 - Анализ спонтанных трат
    3,   # 23 - Раздел "Обучение"
    2,    # 24 - Чтение статей
    0
]

In [44]:
len(predecessors)

26

In [45]:
len(durations)

26

In [46]:
# Рассчёт ранних времён начала и поздних времён окончания
earliest_start, latest_finish = calculate_critical_times(durations, predecessors)

In [49]:
earliest_start

[0,
 0,
 5,
 8,
 12,
 8,
 5,
 5,
 10,
 10,
 10,
 5,
 16,
 18,
 16,
 15,
 9,
 23,
 15,
 18,
 24,
 31,
 23,
 28,
 35,
 37]

In [50]:
latest_finish

[0,
 5,
 13,
 17,
 20,
 20,
 20,
 10,
 21,
 21,
 16,
 25,
 18,
 35,
 27,
 28,
 27,
 31,
 31,
 24,
 31,
 35,
 32,
 35,
 37,
 37]

In [57]:
renewable_demands = [
    [0, 0, 0, 0, 0, 0],  # 0 (фиктивная начальная работа)

    # Блок аутентификации
    [1, 0, 1, 1, 0, 1],  # 1 - Экран регистрации (PM, тестировщик, дизайнер, фронтенд)
    [0, 1, 0, 0, 1, 0],  # 2 - Сервис email (аналитик, бэкенд)
    [0, 0, 1, 1, 0, 1],  # 3 - Экран подтверждения кода (тестировщик, дизайнер, фронтенд)
    [0, 0, 1, 1, 0, 1],  # 4 - Профиль пользователя (тестировщик, дизайнер, фронтенд)
    [0, 0, 1, 0, 1, 0],  # 5 - Аутентификация (тестировщик, бэкенд)

    # Блок финансов
    [1, 1, 0, 1, 0, 1],  # 6 - Экран "Мои финансы" (PM, аналитик, дизайнер, фронтенд)
    [0, 1, 1, 1, 1, 1],  # 7 - Добавление дохода (аналитик, тестировщик, дизайнер, бэкенд, фронтенд)
    [0, 1, 1, 1, 1, 1],  # 8 - Добавление расхода (аналитик, тестировщик, дизайнер, бэкенд, фронтенд)
    [0, 1, 0, 0, 1, 0],  # 9 - Выбор регулярных операций (аналитик, бэкенд)

    # Блок активов и тегов
    [1, 1, 0, 1, 1, 1],  # 10 - Активы и пассивы (PM, аналитик, дизайнер, бэкенд, фронтенд)
    [0, 1, 0, 0, 1, 0],  # 11 - Система тегов (аналитик, бэкенд)
    [0, 0, 1, 0, 0, 1],  # 12 - Добавить тег в процессе (тестировщик, фронтенд)
    [0, 1, 0, 0, 1, 0],  # 13 - Создание напоминания (аналитик, бэкенд)

    # Блок уведомлений и целей
    [0, 0, 1, 0, 1, 1],  # 14 - Пуш-уведомления (тестировщик, бэкенд, фронтенд)
    [1, 1, 0, 1, 1, 1],  # 15 - Планирование крупной цели (PM, аналитик, дизайнер, бэкенд, фронтенд)
    [0, 0, 1, 0, 1, 0],  # 16 - Ежемесячное уведомление (тестировщик, бэкенд)
    [0, 1, 1, 0, 1, 1],  # 17 - Предупреждение о превышении (аналитик, тестировщик, бэкенд, фронтенд)

    # Блок аналитики
    [1, 1, 0, 1, 1, 1],  # 18 - Раздел "Аналитика" (PM, аналитик, дизайнер, бэкенд, фронтенд)
    [0, 0, 1, 0, 1, 1],  # 19 - Круговая диаграмма (тестировщик, бэкенд, фронтенд)
    [0, 0, 1, 0, 1, 1],  # 20 - График доходов/расходов (тестировщик, бэкенд, фронтенд)
    [0, 1, 1, 0, 1, 1],  # 21 - Прогресс по целям (аналитик, тестировщик, бэкенд, фронтенд)
    [0, 1, 1, 0, 1, 0],  # 22 - Анализ спонтанных трат (аналитик, тестировщик, бэкенд)

    # Блок обучения
    [1, 1, 0, 1, 0, 1],  # 23 - Раздел "Обучение" (PM, аналитик, дизайнер, фронтенд)
    [0, 0, 1, 0, 0, 1],   # 24 - Чтение статей (тестировщик, фронтенд)
    [0, 0, 0, 0, 0, 0]
]

In [55]:
len(renewable_demands)

26

In [58]:
renewable_capacities = [1,2,1,1,2,2]

In [59]:
sampler = ActivityListSampler(predecessors)
decoder = ActivityListDecoder()

In [60]:
# Генерируем случайный порядок
random_activity_list = sampler.generate_random()
start_times = decoder.decode(random_activity_list, durations, predecessors, renewable_demands, renewable_capacities)

In [62]:
random_activity_list

[0,
 1,
 2,
 7,
 3,
 9,
 8,
 6,
 4,
 18,
 5,
 10,
 15,
 14,
 12,
 13,
 11,
 19,
 20,
 16,
 22,
 23,
 17,
 21,
 24,
 25]

In [63]:
start_times

[0,
 0,
 5,
 10,
 21,
 24,
 19,
 5,
 14,
 10,
 34,
 8,
 47,
 49,
 40,
 40,
 19,
 67,
 24,
 49,
 55,
 70,
 62,
 67,
 74,
 76]

In [61]:
# Генерируем порядок в соответствии с правилом (по возрастанию поздних времен конца)
heuristic_activity_list = sampler.generate_by_min_rule(lambda j: latest_finish[j])
start_times2 = decoder.decode(random_activity_list, durations, predecessors, renewable_demands, renewable_capacities)

In [64]:
heuristic_activity_list

[0,
 1,
 7,
 2,
 10,
 3,
 12,
 6,
 5,
 4,
 8,
 9,
 19,
 11,
 14,
 16,
 15,
 18,
 20,
 17,
 22,
 13,
 21,
 23,
 24,
 25]

In [65]:
start_times2

[0,
 0,
 5,
 10,
 21,
 24,
 19,
 5,
 14,
 10,
 34,
 8,
 47,
 49,
 40,
 40,
 19,
 67,
 24,
 49,
 55,
 70,
 62,
 67,
 74,
 76]

In [66]:
# Генерируем порядок в соответствии с правилом (по возрастанию поздних времен конца)
heuristic_activity_list3 = sampler.generate_by_max_rule(lambda j: latest_finish[j])
start_times3 = decoder.decode(random_activity_list, durations, predecessors, renewable_demands, renewable_capacities)

In [67]:
heuristic_activity_list3

[0,
 1,
 11,
 16,
 6,
 2,
 5,
 3,
 4,
 15,
 17,
 7,
 8,
 9,
 18,
 10,
 14,
 22,
 23,
 12,
 13,
 19,
 20,
 21,
 24,
 25]

In [68]:
start_times3

[0,
 0,
 5,
 10,
 21,
 24,
 19,
 5,
 14,
 10,
 34,
 8,
 47,
 49,
 40,
 40,
 19,
 67,
 24,
 49,
 55,
 70,
 62,
 67,
 74,
 76]

Дополнение своими эвристиками

In [94]:
from random import expovariate
import numpy as np

In [88]:
class ActivityListSampler:
    def __init__(
            self,
            predecessors: list[list[int]],
            successors: list[list[int]] = None) -> None:
        """
        Args:
            predecessors (list[list[int]]): списки предшествующих работ
            successors (list[list[int]]): списки последующих работ
        """
        self.predecessors = predecessors
        self.size = len(predecessors)
        if not successors:
            successors = successors_by_predecessors(predecessors)
        self.successors = successors

    def _generate(self, func: Callable = None) -> list[int]:
        result = []
        ramain_predecessors = [set(pred) for pred in self.predecessors]
        ready_set = [i for i in range(self.size) if not self.predecessors[i]]

        for _ in range(self.size):
            if not ready_set:
                raise ValueError("Incorrect project network")

            next_activity = func(ready_set) if func else choice(ready_set)
            ready_set.remove(next_activity)
            result.append(next_activity)

            for successor in self.successors[next_activity]:
                ramain_predecessors[successor].remove(next_activity)
                if not ramain_predecessors[successor]:
                    ready_set.append(successor)

        return result

    def generate_random(self) -> list[int]:
        """
        Генерация Activity List случайным образом

        Returns:
            list[int]: Activity List
        """
        return self._generate()

    def generate_by_max_rule(self, rule: Callable) -> list[int]:
        """
        Генерация Activity List упорядочивая по убыванию метрики

        Returns:
            list[int]: Activity List
        """
        def func(data: list[int]):
            return max(data, key=rule)
        return self._generate(func)

    def generate_by_min_rule(self, rule: Callable) -> list[int]:
        """
        Генерация Activity List упорядочивая по возрастанию метрики

        Returns:
            list[int]: Activity List
        """
        def func(data: list[int]):
            return min(data, key=rule)
        return self._generate(func)

    # SLK - по возрастанию общего резерва - значит мин
    def generate_by_SLK(self, latest_finish: list[int], earliest_start: list[int], duration: list[int]) -> list[int]:
        def slack(j):
            return latest_finish[j] - earliest_start[j] - duration[j]
        return self.generate_by_min_rule(slack)

    # FREE - по возрастанию свободного резерва
    # FREE(j) = min(EST(k)) - (EST(j) + duration[j])
    # FREE(j) = min(EST(k)) - EST(j) - duration[j]
    def generate_by_FREE(self, earliest_start: list[int], duration: list[int], successors: list[list[int]]) -> list[int]:
        def free_slack(j):
            if not successors[j]:
                return 0
            return min(earliest_start[succ] for succ in self.successors[j]) - earliest_start[j] - duration[j]
        return self.generate_by_min_rule(free_slack)

    # LST - по возрастанию позднего времени начала
    def generate_by_LST(self, latest_finish: list[int], duration: list[int]) -> list[int]:
        def latest_start(j):
            return latest_finish[j] - duration[j]
        return self.generate_by_min_rule(latest_start)

    # LFT - по возрастанию позднего времени завершения
    def generate_by_LFT(self, latest_finish: list[int]) -> list[int]:
        return self.generate_by_min_rule(lambda j: latest_finish[j])

    def generate_by_exp(self, rule: Callable) -> list[int]:

        def func(data: list[int]):
            return np.random.exponential(scale=1.0, size=None)
        return self._generate(func)

    def generate_by_exp(self, rule: Callable, lambda_param: float = 1.0) -> list[int]:
        """
        Версия с экспоненциальным распределением
        """
        def func(data: list[int]):
            # Генерируем экспоненциально распределенные веса
            weights = [expovariate(lambda_param) for _ in range(len(data))]

            # Выбираем задачу с наименьшим весом (экспоненциальное распределение)
            return data[np.argmin(weights)]

        return self._generate(func)

In [92]:
sampler = ActivityListSampler(predecessors)
decoder = ActivityListDecoder()

In [95]:
# Применение всех эвристик
solutions = {}

# Доп. эвристики
solutions['SLK'] = sampler.generate_by_SLK(latest_finish, earliest_start, durations)
solutions['FREE'] = sampler.generate_by_FREE(latest_finish, earliest_start, durations)
solutions['LST'] = sampler.generate_by_LST(latest_finish, durations)
solutions['LFT'] = sampler.generate_by_LFT(latest_finish)
solutions['EXP'] = sampler.generate_by_exp(lambda j: latest_finish[j])

In [96]:
# Поиск лучшего решения
best_duration = float('inf')
best_solution = None
best_name = ""

for name, activity_list in solutions.items():
    try:
        start_times = decoder.decode(activity_list, durations, predecessors, renewable_demands, renewable_capacities)
        print(name)
        print(start_times)
        makespan = max(start_times[i] + durations[i] for i in range(len(durations)))

        if makespan < best_duration:
            best_duration = makespan
            best_solution = activity_list
            best_name = name
    except:
        continue

print(f"Лучшая эвристика: {best_name} с длительностью {best_duration}")

SLK
[0, 0, 5, 38, 42, 50, 16, 5, 45, 10, 10, 13, 16, 18, 31, 60, 54, 68, 50, 18, 24, 71, 56, 68, 75, 77]
FREE
[0, 0, 5, 11, 15, 18, 8, 22, 60, 27, 27, 5, 45, 47, 33, 33, 9, 65, 65, 47, 53, 75, 40, 45, 79, 81]
LST
[0, 0, 5, 16, 27, 10, 25, 5, 20, 14, 10, 16, 25, 27, 36, 30, 50, 57, 38, 30, 43, 60, 52, 57, 64, 66]
LFT
[0, 0, 5, 16, 22, 10, 20, 5, 25, 14, 10, 16, 20, 22, 36, 30, 43, 52, 38, 30, 45, 60, 55, 60, 64, 66]
EXP
[0, 0, 5, 8, 23, 17, 17, 12, 40, 17, 26, 5, 50, 55, 32, 32, 21, 52, 45, 55, 61, 68, 45, 55, 72, 74]
Лучшая эвристика: LST с длительностью 66


Перерасчет лучшего решения на рабочее время

In [98]:
import datetime
from datetime import timedelta

In [99]:
def calculate_calendar_schedule_simple(start_times: list[int], durations: list[int], start_date: datetime.date):
    """Упрощенный расчет календарного графика"""

    def work_days_to_calendar(work_days: int) -> datetime.date:
        """Конвертирует рабочие дни в календарную дату"""
        calendar_days = work_days
        full_weeks = work_days // 5
        remaining_days = work_days % 5
        total_calendar_days = work_days + full_weeks * 2

        date = start_date + timedelta(days=total_calendar_days)

        # Корректировка если дата попадает на выходные
        while date.weekday() >= 5:  # 5=суббота, 6=воскресенье
            date += timedelta(days=1)
            total_calendar_days += 1

        return date, total_calendar_days

    schedule = []
    for i, work_start in enumerate(start_times):
        start_date_cal, start_cal_days = work_days_to_calendar(work_start)

        # Расчет окончания с учетом длительности
        work_finish = work_start + durations[i]
        finish_date_cal, finish_cal_days = work_days_to_calendar(work_finish)

        schedule.append({
            'task': i,
            'work_start': work_start,
            'work_finish': work_finish,
            'calendar_start': start_date_cal,
            'calendar_finish': finish_date_cal,
            'duration': durations[i],
            'calendar_days': finish_cal_days - start_cal_days
        })

    return schedule

# Расчет календарного графика
calendar_schedule = calculate_calendar_schedule_simple(start_times, durations, datetime.date(2024, 2, 1))

In [103]:
# Определение ключевых этапов на основе реальных задач
milestones = {
    'Начало проекта': 0,
    'Завершение аутентификации': 5,  # задача 5 (механизм аутентификации)
    'Готовность основных финансовых операций': 25,  # задачи 6,7,8 (Мои финансы + доход/расход)
    'Завершение системы уведомлений': 36,  # задача 14 (пуш-уведомления)
    'Готовность аналитики': 57,  # задача 17 (раздел Аналитика)
    'Завершение проекта': 66  # конечная задача
}

print("ПЛАНОВЫЙ ГРАФИК ПРОЕКТА")
print("=" * 60)
print(f"Начало проекта: {start_date.strftime('%d.%m.%Y')}")

# Расчет календарного графика для ключевых этапов
milestone_schedule = []
for milestone_name, work_day in milestones.items():
    # Для каждой вехи получаем календарную дату
    schedule_item = calculate_calendar_schedule_simple([work_day], [0], start_date)[0]
    milestone_schedule.append({
        'name': milestone_name,
        'work_day': work_day,
        'date': schedule_item['calendar_start']
    })

print("\nКЛЮЧЕВЫЕ ЭТАПЫ:")
print("-" * 40)
for milestone in milestone_schedule:
    print(f"{milestone['name']:35}: {milestone['date'].strftime('%d.%m.%Y')} ({milestone['work_day']} раб.день)")

# Расчет общей календарной длительности
project_start = milestone_schedule[0]['date']
project_finish = milestone_schedule[-1]['date']
total_calendar_days = (project_finish - project_start).days

print(f"\nОБЩАЯ ДЛИТЕЛЬНОСТЬ:")
print(f"- Рабочих дней: 66")
print(f"- Календарных дней: {total_calendar_days}")
print(f"- Плановое окончание: {project_finish.strftime('%d.%m.%Y')}")

print("\nДЕТАЛЬНЫЙ ГРАФИК ЗАДАЧ:")
print("-" * 60)
for task in calendar_schedule:
    if task['duration'] > 0:  # пропускаем фиктивные задачи
        print(f"Задача {task['task']:2}: {task['calendar_start'].strftime('%d.%m.%Y')} - {task['calendar_finish'].strftime('%d.%m.%Y')} "
              f"({task['duration']} дн.)")

ПЛАНОВЫЙ ГРАФИК ПРОЕКТА
Начало проекта: 01.02.2024

КЛЮЧЕВЫЕ ЭТАПЫ:
----------------------------------------
Начало проекта                     : 01.02.2024 (0 раб.день)
Завершение аутентификации          : 08.02.2024 (5 раб.день)
Готовность основных финансовых операций: 07.03.2024 (25 раб.день)
Завершение системы уведомлений     : 22.03.2024 (36 раб.день)
Готовность аналитики               : 22.04.2024 (57 раб.день)
Завершение проекта                 : 03.05.2024 (66 раб.день)

ОБЩАЯ ДЛИТЕЛЬНОСТЬ:
- Рабочих дней: 66
- Календарных дней: 92
- Плановое окончание: 03.05.2024

ДЕТАЛЬНЫЙ ГРАФИК ЗАДАЧ:
------------------------------------------------------------
Задача  1: 01.02.2024 - 08.02.2024 (5 дн.)
Задача  2: 08.02.2024 - 12.02.2024 (3 дн.)
Задача  3: 12.02.2024 - 19.02.2024 (4 дн.)
Задача  4: 04.03.2024 - 08.03.2024 (3 дн.)
Задача  5: 26.02.2024 - 01.03.2024 (4 дн.)
Задача  6: 26.02.2024 - 26.02.2024 (2 дн.)
Задача  7: 19.02.2024 - 26.02.2024 (5 дн.)
Задача  8: 28.03.2024 - 04.04.2024

In [104]:
# Группировка задач по этапам проекта
project_stages = {
    'Этап 1: Аутентификация и профиль': [1, 2, 3, 4, 5],
    'Этап 2: Основные финансовые операции': [6, 7, 8, 9],
    'Этап 3: Система тегов и категорий': [10, 11, 12],
    'Этап 4: Уведомления и напоминания': [13, 14],
    'Этап 5: Планирование целей': [15, 16],
    'Этап 6: Аналитика и отчетность': [17, 18, 19, 20],
    'Этап 7: Дополнительные функции': [21, 22, 23, 24]
}

print("\nЭТАПЫ ПРОЕКТА С ДАТАМИ:")
print("=" * 50)

for stage_name, task_ids in project_stages.items():
    # Находим даты начала и окончания этапа
    stage_start_times = [start_times[task_id] for task_id in task_ids]
    stage_durations = [durations[task_id] for task_id in task_ids]
    stage_finish_times = [start_times[task_id] + durations[task_id] for task_id in task_ids]

    stage_start_day = min(stage_start_times)
    stage_finish_day = max(stage_finish_times)

    # Конвертируем в календарные даты
    start_calendar = calculate_calendar_schedule_simple([stage_start_day], [0], start_date)[0]['calendar_start']
    finish_calendar = calculate_calendar_schedule_simple([stage_finish_day], [0], start_date)[0]['calendar_start']

    print(f"{stage_name:35}: {start_calendar.strftime('%d.%m.%Y')} - {finish_calendar.strftime('%d.%m.%Y')}")
    print(f"{'':35}  ({stage_start_day}-{stage_finish_day} раб.дни)")


ЭТАПЫ ПРОЕКТА С ДАТАМИ:
Этап 1: Аутентификация и профиль   : 01.02.2024 - 14.03.2024
                                     (0-30 раб.дни)
Этап 2: Основные финансовые операции: 08.02.2024 - 11.03.2024
                                     (5-27 раб.дни)
Этап 3: Система тегов и категорий  : 15.02.2024 - 15.03.2024
                                     (10-31 раб.дни)
Этап 4: Уведомления и напоминания  : 11.03.2024 - 01.04.2024
                                     (27-43 раб.дни)
Этап 5: Планирование целей         : 14.03.2024 - 15.04.2024
                                     (30-52 раб.дни)
Этап 6: Аналитика и отчетность     : 14.03.2024 - 06.05.2024
                                     (30-67 раб.дни)
Этап 7: Дополнительные функции     : 15.04.2024 - 06.05.2024
                                     (52-67 раб.дни)
